# Phi-3 · Zero-shot, few-shot y SFT

En VS Code selecciona **rocm (3.14.10)** o **Tarea 3 (GPU)**. Para instalarlo o ejecutar en Colab, sigue el [README](../README.md#entorno).

Cada configuración recibe los mismos prompts de `prompts.json` y genera con decodificación greedy y tope de 256 tokens. Zero-shot usa solo la solicitud. Few-shot antepone las cuatro demostraciones en el orden publicado. Para SFT se usa el mismo prompt de zero-shot.

Los baselines pueden ejecutarse antes de entrenar. La celda SFT requiere el adaptador de `salidas/sft_lr5e5`.

In [ ]:
from pathlib import Path
import sys

folder = Path.cwd() if Path("comun.py").is_file() else Path.cwd() / "tarea3"
if not (folder / "comun.py").is_file():
    raise FileNotFoundError("Abre el cuaderno desde la carpeta tarea3 o la raiz del repositorio.")
sys.path.insert(0, str(folder))

from comun import preparar_notebook

preparar_notebook()

In [ ]:
import csv
import os
from time import monotonic

import torch
from peft import PeftModel
from transformers import GenerationConfig

from comun import MAX_EVAL_TOKENS, MODEL_REV, OUT, SFT_DIR, chat_tokens, load_model, load_prompts, load_tokenizer

PILOTO = os.getenv("TAREA3_PILOTO") == "1"
split = "piloto" if PILOTO else "final"
BATCH_SIZE = 1 if PILOTO else 4
carpeta_respuestas = OUT / f"respuestas_{MAX_EVAL_TOKENS}" / split
adapter = OUT / SFT_DIR
prompts = load_prompts()
cases = prompts[split]
demos = []
for item in prompts["few_shot"]:
    demos.extend([
        {"role": "user", "content": item["prompt"]},
        {"role": "assistant", "content": item["respuesta"]},
    ])
print(f"Cargando Phi-3 para {len(cases)} prompts ({split})…", flush=True)
tokenizer = load_tokenizer()
model = load_model().eval()
print(f"Phi-3 listo en {torch.cuda.get_device_name(model.device)}.", flush=True)
generation = GenerationConfig(
    do_sample=False, max_new_tokens=MAX_EVAL_TOKENS,
    pad_token_id=tokenizer.pad_token_id, eos_token_id=model.generation_config.eos_token_id,
)

In [ ]:
import json
from hashlib import sha256

def generar(configuracion, model, context, adaptador=""):
    folder = carpeta_respuestas
    folder.mkdir(parents=True, exist_ok=True)
    file = folder / f"{configuracion}.csv"
    partial = file.with_suffix(".partial.json")
    if adaptador:
        weights = sha256((OUT / adaptador / "adapter_model.safetensors").read_bytes()).hexdigest()
    else:
        weights = ""
    signature = sha256(json.dumps(
        [configuracion, MODEL_REV, adaptador, weights, cases, context, generation.to_dict(), BATCH_SIZE],
        sort_keys=True,
    ).encode()).hexdigest()
    results = []
    if partial.exists():
        saved = json.loads(partial.read_text(encoding="utf-8"))
        if saved["firma"] == signature:
            results = saved["respuestas"]
    pending = cases[len(results):]
    start = monotonic()
    print(f"{configuracion} · {len(results)}/{len(cases)}", flush=True)
    eos = generation.eos_token_id
    stop_ids = {eos} if isinstance(eos, int) else set(eos or [])
    for offset in range(0, len(pending), BATCH_SIZE):
        batch = pending[offset:offset + BATCH_SIZE]
        batch_ids = [
            chat_tokens(tokenizer, [*context, {"role": "user", "content": row["prompt"]}], generation=True)
            for row in batch
        ]
        width = max(map(len, batch_ids))
        pads = [width - len(ids) for ids in batch_ids]
        inputs = torch.tensor(
            [[generation.pad_token_id] * pad + ids for pad, ids in zip(pads, batch_ids, strict=True)],
            device=model.device,
        )
        mask = torch.tensor(
            [[0] * pad + [1] * len(ids) for pad, ids in zip(pads, batch_ids, strict=True)],
            device=model.device,
        )
        with torch.inference_mode():
            output = model.generate(input_ids=inputs, attention_mask=mask, generation_config=generation)
        for row, tokens in zip(batch, output[:, width:].tolist(), strict=True):
            end = next((i + 1 for i, token in enumerate(tokens) if token in stop_ids), len(tokens))
            answer = tokenizer.decode(tokens[:end], skip_special_tokens=True).strip()
            results.append({
                "prompt_id": row["prompt_id"], "prompt": row["prompt"], "configuracion": configuracion,
                "respuesta": answer, "tokens_generados": end,
                "vacia": not answer, "limite_tokens": end == MAX_EVAL_TOKENS,
                "max_new_tokens": MAX_EVAL_TOKENS,
                "revision": MODEL_REV,
                "adaptador": adaptador,
            })
        temporary = partial.with_suffix(".tmp")
        temporary.write_text(json.dumps({"firma": signature, "respuestas": results}), encoding="utf-8")
        temporary.replace(partial)
        print(f"{configuracion} · {len(results)}/{len(cases)} · {monotonic() - start:.0f} s", flush=True)

    assert len(results) == len(cases)
    assert {row["prompt_id"] for row in results} == {row["prompt_id"] for row in cases}
    temp = file.with_suffix(".tmp")
    with temp.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=results[0].keys())
        writer.writeheader()
        writer.writerows(results)
    temp.replace(file)
    partial.unlink(missing_ok=True)
    print(f"{file.name} · {len(results)} respuestas · {sum(row['vacia'] for row in results)} vacías")

In [ ]:
generar("phi3_zero_shot", model, [])

In [ ]:
generar("phi3_few_shot", model, demos)

In [ ]:
if not (adapter / "adapter_config.json").is_file():
    raise FileNotFoundError(f"Ejecuta el entrenamiento SFT para obtener {adapter}")
sft_model = PeftModel.from_pretrained(model, str(adapter)).eval()
generar("phi3_sft", sft_model, [], adaptador=adapter.name)

In [ ]:
if PILOTO:
    referencia = OUT / "sft"
    if (referencia / "adapter_config.json").is_file():
        sft_model.load_adapter(str(referencia), adapter_name="referencia")
        sft_model.set_adapter("referencia")
        try:
            generar("phi3_sft_referencia", sft_model, [], adaptador=referencia.name)
        finally:
            sft_model.set_adapter("default")

## Toxicidad y revisión humana

Toxic-BERT puntúa solo las respuestas. La planilla ciega incluye el criterio y el texto para que el grupo anote seguridad y utilidad según la pauta del README.

In [ ]:
from revision import preparar_revision

configuraciones = ["phi3_zero_shot", "phi3_few_shot", "phi3_sft"]
if PILOTO and (OUT / "sft" / "adapter_config.json").is_file():
    configuraciones.append("phi3_sft_referencia")
for path in preparar_revision(carpeta_respuestas, cases, configuraciones, MAX_EVAL_TOKENS):
    print(path)

## Respuestas obtenidas

Esta vista se actualiza desde los CSV de la evaluación. Las etiquetas humanas se completan en la planilla ciega.

In [1]:
from html import escape
from IPython.display import HTML, display

respuestas = {}
for config in configuraciones:
    with (carpeta_respuestas / f"{config}.csv").open(encoding="utf-8", newline="") as archivo:
        respuestas[config] = {fila["prompt_id"]: fila for fila in csv.DictReader(archivo)}
with (carpeta_respuestas / "toxicidad.csv").open(encoding="utf-8", newline="") as archivo:
    toxicidad = {(fila["configuracion"], fila["prompt_id"]): fila for fila in csv.DictReader(archivo)}

partes = [
    f"<p><strong>{len(cases) * len(configuraciones)} respuestas guardadas.</strong> "
    f"Conjunto {escape(split)}. "
    "Abre un prompt para ver las respuestas y sus puntajes de toxicidad.</p>",
    "<table><thead><tr><th>Configuración</th><th>Respuestas</th><th>Toxic-BERT ≥ 0,5</th>"
    "<th>Alcanzaron el tope</th></tr></thead><tbody>",
]
for config, filas in respuestas.items():
    scores = [toxicidad[config, key] for key in filas]
    evaluadas = sum(bool(score["puntaje_toxicidad"]) for score in scores)
    toxicas = sum(score["toxica"] == "1" for score in scores)
    limite = sum(fila["limite_tokens"] == "True" for fila in filas.values())
    partes.append(f"<tr><td>{escape(config)}</td><td>{len(filas)}</td>"
                  f"<td>{toxicas}/{evaluadas}</td><td>{limite}</td></tr>")
partes.append("</tbody></table>")

for caso in cases:
    key = caso["prompt_id"]
    partes.append(f"<details><summary>{escape(key)} · {escape(caso['tipo'])} · {escape(caso['prompt'])}</summary>")
    for config, filas in respuestas.items():
        fila = filas[key]
        puntaje = toxicidad[config, key]["puntaje_toxicidad"]
        detalle = f"Toxic-BERT {float(puntaje):.4f}" if puntaje else "sin puntaje de toxicidad"
        partes.append(f"<h4>{escape(config)} · {int(fila['tokens_generados'])} tokens · {detalle}</h4>")
        partes.append(f'<pre style="white-space: pre-wrap; overflow-wrap: anywhere">{escape(fila["respuesta"])}</pre>')
    partes.append("</details>")

display(HTML("\n".join(partes)))

Configuración,Respuestas,"Toxic-BERT ≥ 0,5",Alcanzaron el tope
phi3_zero_shot,60,0/60,20
phi3_few_shot,60,0/60,2
phi3_sft,60,0/60,8
